# Walmart Store Sales Forecasting — LightGBM Baseline

This notebook trains a simple LightGBM baseline model. It is intentionally smaller than `model_experiment_LightGBM.ipynb`:

- no custom feature engineering
- no lag or rolling features
- no historical target aggregates
- no feature selection
- no Optuna search
- no Model Registry registration

The only preprocessing here is what is needed to train LightGBM on the raw merged Walmart tables.


In [ ]:
%pip install -q "lightgbm>=4,<5" "wandb>=0.19,<1" "pandas>=2.2,<3" "numpy>=1.26,<3" "matplotlib>=3.8,<4"


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print(f"Drive mount skipped: {exc}")


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wandb

WANDB_ENTITY = "kende23-n-a"
WANDB_PROJECT = "Walmart-Recruiting---Store-Sales-Forecasting"
VALIDATION_WEEKS = 32
HOLIDAY_WEIGHT = 5
SEED = 42

DATA_DIR_CANDIDATES = [
    Path("/content/drive/MyDrive/walmart_competition_data"),
    Path("/content/drive/My Drive/walmart_competition_data"),
    Path("/content/walmart_competition_data"),
    Path("data"),
    Path("../../data"),
]
OUTPUT_DIR = Path("/content/artifacts/lightgbm_baseline") if Path("/content").exists() else Path("artifacts/lightgbm_baseline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def resolve_data_dir(candidates):
    required = ["train.csv", "features.csv", "stores.csv"]
    for candidate in candidates:
        if all((candidate / filename).exists() for filename in required):
            return candidate
    raise FileNotFoundError(
        "Could not find Walmart data directory with train.csv, features.csv, and stores.csv. "
        f"Checked: {[str(path) for path in candidates]}"
    )


def weighted_mae(y_true, y_pred, is_holiday, holiday_weight=5):
    weights = np.where(np.asarray(is_holiday).astype(bool), holiday_weight, 1.0)
    return float(np.sum(np.abs(np.asarray(y_true) - np.asarray(y_pred)) * weights) / np.sum(weights))


DATA_DIR = resolve_data_dir(DATA_DIR_CANDIDATES)
print(f"Using data directory: {DATA_DIR}")


## Load and merge raw data

The baseline uses the standard Walmart tables merged by `Store`, `Date`, and `IsHoliday`. The authoritative holiday flag comes from `train.csv`.


In [ ]:
train_raw = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["Date"])
features_raw = pd.read_csv(DATA_DIR / "features.csv", parse_dates=["Date"])
stores_raw = pd.read_csv(DATA_DIR / "stores.csv")

train_df = train_raw.merge(stores_raw, on="Store", how="left")
train_df = train_df.merge(
    features_raw,
    on=["Store", "Date", "IsHoliday"],
    how="left",
)

train_df = train_df.sort_values(["Date", "Store", "Dept"]).reset_index(drop=True)

profile = {
    "rows": int(len(train_df)),
    "stores": int(train_df["Store"].nunique()),
    "departments": int(train_df["Dept"].nunique()),
    "weeks": int(train_df["Date"].nunique()),
    "start_date": str(train_df["Date"].min().date()),
    "end_date": str(train_df["Date"].max().date()),
    "missing_values": int(train_df.isna().sum().sum()),
}
display(pd.Series(profile, name="value").to_frame())
display(train_df.head())


## Chronological validation split

The validation set is the last 32 weekly dates, matching the later LightGBM experiment split.


In [ ]:
validation_dates = np.sort(train_df["Date"].unique())[-VALIDATION_WEEKS:]
train_part = train_df.loc[~train_df["Date"].isin(validation_dates)].copy()
val_part = train_df.loc[train_df["Date"].isin(validation_dates)].copy()

split_summary = {
    "validation_weeks": VALIDATION_WEEKS,
    "train_rows": int(len(train_part)),
    "validation_rows": int(len(val_part)),
    "train_start": str(train_part["Date"].min().date()),
    "train_end": str(train_part["Date"].max().date()),
    "validation_start": str(val_part["Date"].min().date()),
    "validation_end": str(val_part["Date"].max().date()),
    "validation_unique_weeks": int(val_part["Date"].nunique()),
}
display(pd.Series(split_summary, name="value").to_frame())


## Minimal preprocessing

This is not feature engineering. It only makes the merged raw columns usable by LightGBM:

- drop `Date`, because LightGBM cannot use pandas datetime directly here
- fill missing markdown values with `0`
- fill missing numeric external values with the training median
- cast `Type` to pandas category so LightGBM can treat it as categorical
- keep `Store`, `Dept`, and `IsHoliday` as simple raw identifiers/flags


In [ ]:
TARGET = "Weekly_Sales"
DROP_COLS = ["Date", TARGET]
MARKDOWN_COLS = ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]
NUMERIC_EXTERNAL_COLS = ["Temperature", "Fuel_Price", "CPI", "Unemployment"]
CATEGORICAL_COLS = ["Type"]


def prepare_baseline_features(frame, numeric_medians=None):
    prepared = frame.drop(columns=DROP_COLS, errors="ignore").copy()

    for col in MARKDOWN_COLS:
        if col in prepared.columns:
            prepared[col] = prepared[col].fillna(0.0)

    if numeric_medians is None:
        numeric_medians = {
            col: prepared[col].median()
            for col in NUMERIC_EXTERNAL_COLS
            if col in prepared.columns
        }

    for col, median_value in numeric_medians.items():
        prepared[col] = prepared[col].fillna(median_value)

    if "IsHoliday" in prepared.columns:
        prepared["IsHoliday"] = prepared["IsHoliday"].astype("int8")

    for col in CATEGORICAL_COLS:
        if col in prepared.columns:
            prepared[col] = prepared[col].astype("category")

    return prepared, numeric_medians


X_train, numeric_medians = prepare_baseline_features(train_part)
X_val, _ = prepare_baseline_features(val_part, numeric_medians=numeric_medians)
y_train = train_part[TARGET]
y_val = val_part[TARGET]

categorical_features = [col for col in CATEGORICAL_COLS if col in X_train.columns]
sample_weights_train = np.where(train_part["IsHoliday"].astype(bool), HOLIDAY_WEIGHT, 1.0)
sample_weights_val = np.where(val_part["IsHoliday"].astype(bool), HOLIDAY_WEIGHT, 1.0)

feature_summary = {
    "feature_count": int(X_train.shape[1]),
    "categorical_features": categorical_features,
    "train_missing_values": int(X_train.isna().sum().sum()),
    "validation_missing_values": int(X_val.isna().sum().sum()),
}
display(pd.Series(feature_summary, name="value").to_frame())
display(X_train.head())


## Train baseline LightGBM


In [ ]:
try:
    from google.colab import userdata
    wandb_api_key = userdata.get("WANDB_API_KEY")
except Exception:
    wandb_api_key = os.environ.get("WANDB_API_KEY")

wandb.login(key=wandb_api_key, relogin=False) if wandb_api_key else wandb.login()

baseline_params = {
    "objective": "regression_l1",
    "metric": "mae",
    "boosting_type": "gbdt",
    "n_estimators": 500,
    "learning_rate": 0.05,
    "num_leaves": 64,
    "max_depth": -1,
    "min_child_samples": 40,
    "subsample": 0.9,
    "subsample_freq": 1,
    "colsample_bytree": 0.9,
    "reg_alpha": 0.0,
    "reg_lambda": 0.0,
    "random_state": SEED,
    "n_jobs": -1,
    "verbose": -1,
}

run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    job_type="baseline_train",
    name="LightGBM_Baseline",
    tags=["lightgbm", "baseline", "time-split"],
    config={
        **profile,
        **split_summary,
        **feature_summary,
        "holiday_weight": HOLIDAY_WEIGHT,
        **baseline_params,
    },
)

baseline_model = lgb.LGBMRegressor(**baseline_params)
baseline_model.fit(
    X_train,
    y_train,
    sample_weight=sample_weights_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    eval_names=["train", "validation"],
    eval_sample_weight=[sample_weights_train, sample_weights_val],
    eval_metric="mae",
    categorical_feature=categorical_features,
    callbacks=[lgb.log_evaluation(period=50)],
)


## Evaluate and log baseline result


In [ ]:
val_pred = baseline_model.predict(X_val)
val_pred = np.clip(val_pred, 0.0, None)

validation_weighted_mae = weighted_mae(y_val, val_pred, val_part["IsHoliday"], holiday_weight=HOLIDAY_WEIGHT)
validation_mae = float(np.mean(np.abs(y_val.to_numpy() - val_pred)))
validation_rmse = float(np.sqrt(np.mean((y_val.to_numpy() - val_pred) ** 2)))

metrics = {
    "validation/weighted_mae": validation_weighted_mae,
    "validation/mae": validation_mae,
    "validation/rmse": validation_rmse,
}

feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": baseline_model.feature_importances_,
}).sort_values("importance", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_val, val_pred, alpha=0.25, s=8)
axes[0].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], "r--", lw=2)
axes[0].set_title("Baseline LightGBM: actual vs predicted")
axes[0].set_xlabel("Actual Weekly_Sales")
axes[0].set_ylabel("Predicted Weekly_Sales")

plot_importance = feature_importance.head(20).sort_values("importance")
axes[1].barh(plot_importance["feature"], plot_importance["importance"])
axes[1].set_title("Top baseline feature importances")
plt.tight_layout()

run.log({
    **metrics,
    "model/feature_importance": wandb.Table(dataframe=feature_importance),
    "plots/baseline_diagnostics": wandb.Image(fig),
})
run.summary.update(metrics)
run.summary["best_validation_weighted_mae"] = validation_weighted_mae
run.finish()

print("Baseline LightGBM validation metrics:")
for key, value in metrics.items():
    print(f"  {key}: {value:.4f}")

display(feature_importance.head(25))


## Notes

Use this baseline result as the comparison point for `model_experiment_LightGBM.ipynb`. The engineered notebook should improve over this baseline after adding calendar, holiday, markdown, interaction, lag/rolling, aggregate, feature-selection, and Optuna tuning steps.
